<a href="https://colab.research.google.com/github/zencolab/colab/blob/main/comfyui_video0430.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# Cell 1: 基础环境与插件管理器 (内置通用安装函数)
# ==========================================
import os, subprocess
print("=== 🚀 开始安装 ComfyUI 基础与图像模块 ===")
%cd /content

if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI
else:
    !cd ComfyUI && git pull

%cd /content/ComfyUI
!pip install -q -r requirements.txt huggingface_hub hf_transfer

# 定义通用节点安装函数以复用代码
def install_node(repo_url):
    folder_name = repo_url.split('/')[-1].replace('.git', '')
    target_path = f"/content/ComfyUI/custom_nodes/{folder_name}"
    if not os.path.exists(target_path):
        !git clone {repo_url} {target_path}
        if os.path.exists(f"{target_path}/requirements.txt"):
            !pip install -q -r {target_path}/requirements.txt
    else:
        !cd {target_path} && git pull

# 安装基础核心插件
base_nodes = [
    "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "https://github.com/rgthree/rgthree-comfy.git"
]
for node in base_nodes:
    install_node(node)
print("\n✅ Cell 1 安装完成！")

In [ ]:
# ==========================================
# Cell 2: 视频生成与处理扩展模块
# ==========================================
import subprocess
print("=== 🎬 开始安装视频处理模块 ===")

if subprocess.call(["which", "ffmpeg"], stdout=subprocess.DEVNULL) != 0:
    !apt update -qq && apt -y install ffmpeg -qq

# 调用 Cell 1 的通用函数快速安装
video_nodes = [
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "https://github.com/kijai/ComfyUI-KJNodes.git",
    "https://github.com/Lightricks/ComfyUI-LTXVideo.git"
]
for node in video_nodes:
    install_node(node)

print("\n✅ Cell 2 视频处理模块安装完成！")

In [ ]:
# ==========================================
# Cell 3: 补充安装缺失的预处理与功能节点
# ==========================================
print("=== 🚀 开始补充安装缺失的功能节点 ===")

missing_nodes = [
    "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
    "https://github.com/cubiq/ComfyUI_essentials.git",
    "https://github.com/yuvraj108c/ComfyUI-Video-Depth-Anything.git",
    "https://github.com/ai-shizuka/comfyui-tbox.git",
    "https://github.com/bollerdomi/ComfyUI-load-lora-from-url.git"
]
for node in missing_nodes:
    install_node(node)

# 补充视觉库依赖
!pip install -q opencv-python-headless scikit-image onnxruntime-gpu

print("\n🎉 Cell 3 缺失节点已全部安装就绪！")

In [ ]:
# ==========================================
# Cell 4: 终极模型多线程极速同步 (极大加快下载)
# ==========================================
import os
from google.colab import userdata
from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except:
    raise Exception("❌ 请在左侧 '🔑 Secrets' 中添加 'HF_TOKEN'")

downloads = [
    {"repo_id": "Lightricks/LTX-2.3-fp8", "filename": "ltx-2.3-22b-dev-fp8.safetensors", "local_dir": "/content/ComfyUI/models/checkpoints"},
    {"repo_id": "Lightricks/LTX-2.3", "filename": "ltx-2.3-22b-distilled-lora-384.safetensors", "local_dir": "/content/ComfyUI/models/loras"},
    {"repo_id": "Lightricks/LTX-2.3", "filename": "ltx-2.3-spatial-upscaler-x2-1.1.safetensors", "local_dir": "/content/ComfyUI/models/latent_upscale_models"},
    {"repo_id": "Comfy-Org/ltx-2", "filename": "split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors", "local_dir": "/content/ComfyUI/models/text_encoders"},
    {"repo_id": "Comfy-Org/ltx-2.3", "filename": "split_files/loras/ltx-2.3-id-lora-talkvid-3k.safetensors", "local_dir": "/content/ComfyUI/models/loras"},
    {"repo_id": "Lightricks/LTX-2.3-22b-IC-LoRA-Union-Control", "filename": "ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors", "local_dir": "/content/ComfyUI/models/loras"}
]

def download_model(task):
    try:
        hf_hub_download(repo_id=task["repo_id"], filename=task["filename"], local_dir=task["local_dir"], repo_type="model")
        print(f"✅ 完成: {task['filename']}")
    except Exception as e:
        print(f"❌ 失败 {task['filename']}: {e}")

print(f"🚀 开始启动多线程并发下载 {len(downloads)} 个模型...")
with ThreadPoolExecutor(max_workers=6) as executor:
    executor.map(download_model, downloads)

print("\n🎉 模型并发同步全部完成！")

In [ ]:
# ==========================================
# Cell 5: FRP 内网穿透配置 (流式解压极速部署)
# ==========================================
import os
from google.colab import userdata

try:
    VPS_IP = userdata.get('VPS_IP')
    FRP_TOKEN = userdata.get('FRP_TOKEN')
except:
    raise Exception("❌ 请检查 'VPS_IP' 和 'FRP_TOKEN' 密钥是否配置！")

# 极速流式下载解压
if not os.path.exists("/content/frp_0.56.0_linux_amd64"):
    !wget -qO- https://github.com/fatedier/frp/releases/download/v0.56.0/frp_0.56.0_linux_amd64.tar.gz | tar -xz -C /content

frpc_conf = f"""
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui_web"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8080
"""
with open("/content/frp_0.56.0_linux_amd64/frpc.toml", "w") as f:
    f.write(frpc_conf.strip())

print("✅ FRP 极速部署并配置完毕！")

In [ ]:
# ==========================================
# 四、启动 FRP 和 ComfyUI (重启界面时运行),不注册，启动快！访问域名
# ==========================================
import subprocess
import threading
import os
import configparser

import time

# --- 新增：防断开保活机制 ---
def keep_alive():
    while True:
        time.sleep(300)  # 每 5 分钟
        print("\n[Keep-Alive] 保持 Colab 连接活跃中...")

print("⏳ 正在启动防断开后台保活线程...")
threading.Thread(target=keep_alive, daemon=True).start()
# --------------------------

# 1. 在后台新线程启动 FRP
def start_frpc():
    subprocess.run(["/content/frp_0.56.0_linux_amd64/frpc", "-c", "/content/frp_0.56.0_linux_amd64/frpc.toml"])

print("⏳ 正在后台唤起 FRP 穿透服务...")
threading.Thread(target=start_frpc, daemon=True).start()

print("============================================================")
print("✅ FRP 穿透已就绪！")
# 这里已修改为你的指定域名（保留了 8080 端口，如果你的服务端用 Nginx 做了 80 端口反代，可以把 :8080 删掉）
print("👉 启动完成后，请通过浏览器访问: http://cjp.usdream.dpdns.org:8080")
print("============================================================\n")

# ---------------------------------------------------------
# 新增：彻底拦截 ComfyUI-Manager 启动时的联网 Fetch 行为
# ---------------------------------------------------------
print("⏳ 正在配置 Manager 网络模式以跳过 Fetch...")
# 兼容所有版本的配置文件路径 (特别是最新的 __manager 路径)
manager_config_paths = [
    "/content/ComfyUI/user/__manager/config.ini",               # 最新版 V3.39+ 配置路径
    "/content/ComfyUI/user/default/ComfyUI-Manager/config.ini", # 较新版配置路径
    "/content/ComfyUI/custom_nodes/ComfyUI-Manager/config.ini"  # 老旧版配置路径
]

for config_path in manager_config_paths:
    os.makedirs(os.path.dirname(config_path), exist_ok=True)
    config = configparser.ConfigParser()

    if os.path.exists(config_path):
        config.read(config_path)

    if 'default' not in config:
        config['default'] = {}

    # 将网络模式改为 private
    config['default']['network_mode'] = 'private'

    with open(config_path, 'w') as f:
        config.write(f)

print("✅ 已成功切断 Fetch ComfyRegistry Data 流程！\n")
# ---------------------------------------------------------

# 2. 启动 ComfyUI 主程序
print("⏳ 正在启动 ComfyUI 主进程...\n")
%cd /content/ComfyUI
!python main.py --dont-print-server